In [ ]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
import time

load_dotenv()

TMDB_KEY = os.getenv("TMDB_API_KEY")
OMDB_KEY = os.getenv("OMDB_API_KEY")

print("✅ Ready")

In [ ]:
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import os

In [ ]:
def get_movies_by_language(language, region, years=[2023, 2024, 2025]):
    movies = []
    
    for year in years:
        print(f"  Fetching {region} movies from {year}...")
        
        for page in range(1, 6):  # 5 pages = 100 movies
            url = f"https://api.themoviedb.org/3/discover/movie"
            params = {
                "api_key":               TMDB_KEY,
                "with_original_language": language,
                "primary_release_year":  year,
                "sort_by":               "popularity.desc",
                "page":                  page
            }
            r = requests.get(url, params=params).json()
            
            if not r.get("results"):
                break
            
            for movie in r["results"]:
                movies.append({
                    "title":   movie["title"],
                    "year":    year,
                    "region":  region,
                    "tmdb_id": movie["id"],
                    "popularity": movie.get("popularity", 0)
                })
            time.sleep(0.1)
    
    return movies

# Fetch all industries
all_movies = []

industries = [
    ("en", "Hollywood"),
    ("hi", "Bollywood"),
    ("ta", "Kollywood"),
    ("te", "Tollywood"),
    ("ml", "Mollywood"),
]

for lang, region in industries:
    print(f"\n🎬 Fetching {region}...")
    movies = get_movies_by_language(lang, region)
    all_movies.extend(movies)
    print(f"  → {len(movies)} movies found")

movies_df = pd.DataFrame(all_movies).drop_duplicates(subset="tmdb_id")
print(f"\n✅ Total: {len(movies_df)} movies across all industries")
movies_df.head(10)

In [ ]:


#get imdb from tmdb
def get_imdb_id(tmdb_id):
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}/external_ids"
    params = {"api_key": TMDB_KEY}
    r = requests.get(url, params=params).json()
    return r.get("imdb_id")

print("Getting IMDb IDs from TMDB...")
imdb_ids = []

for i, row in movies_df.iterrows():
    imdb_id = get_imdb_id(row["tmdb_id"])
    imdb_ids.append(imdb_id)
    if i % 50 == 0:
        print(f"  → {i}/{len(movies_df)} done")
    time.sleep(0.1)

movies_df["imdb_id"] = imdb_ids
movies_df = movies_df[movies_df["imdb_id"].notna()]
print(f"✅ {len(movies_df)} movies with valid IMDb IDs")

In [ ]:
def get_omdb_ratings(imdb_id):
    url = f"http://www.omdbapi.com/?i={imdb_id}&apikey={OMDB_KEY}"
    r = requests.get(url).json()
    
    if r.get("Response") == "False":
        return None, None, None
    
    imdb = r.get("imdbRating")
    imdb = float(imdb) if imdb and imdb != "N/A" else None
    
    rt = next((x["Value"] for x in r.get("Ratings", [])
               if x["Source"] == "Rotten Tomatoes"), None)
    
    genre = r.get("Genre")
    
    return imdb, rt, genre

print("Fetching IMDb + RT ratings from OMDB...")
imdb_ratings, rt_scores, genres = [], [], []

for i, row in movies_df.iterrows():
    imdb, rt, genre = get_omdb_ratings(row["imdb_id"])
    imdb_ratings.append(imdb)
    rt_scores.append(rt)
    genres.append(genre)
    
    if i % 20 == 0:
        print(f"  → {i}/{len(movies_df)} | {row['title']} | IMDb: {imdb} | RT: {rt}")
    time.sleep(0.2)

movies_df["imdb_rating"] = imdb_ratings
movies_df["rt_score"]    = rt_scores
movies_df["genre"]       = genres

# Keep only movies with valid ratings
movies_df = movies_df[movies_df["imdb_rating"].notna()]
print(f"\n✅ {len(movies_df)} movies with valid IMDb ratings")
movies_df.to_csv("data/movies_master.csv", index=False)
movies_df.head(10)

In [ ]:
'''# Test OMDB key
import requests
from dotenv import load_dotenv
import os

load_dotenv()


print(f"Key loaded: {OMDB_KEY}")

# Test with a known movie
r = requests.get(f"http://www.omdbapi.com/?t=Barbie&apikey={OMDB_KEY}").json()
print(r)'''

In [ ]:
# Check actual column names first
print(movies_df.columns.tolist())

In [ ]:
# Show movies where we have all 3 ratings
complete = movies_df[
    movies_df["imdb_rating"].notna() & 
    movies_df["rt_score"].notna()
].copy()

print(f"✅ Complete data for {len(complete)} movies\n")
complete[["title", "region", "imdb_rating", "rt_score", "genre"]].head(300)

In [ ]:
from snowflake.connector.pandas_tools import write_pandas
import snowflake.connector

conn = snowflake.connector.connect(
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    warehouse="COMPUTE_WH",
    database="MOVIE_ANALYTICS",
    schema="RAW"
)

# Create table if not exists
cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE IF NOT EXISTS raw.movies_master (
        tmdb_id      STRING,
        imdb_id      STRING,
        title        STRING,
        year         STRING,
        genre        STRING,
        region       STRING,
        imdb_rating  FLOAT,
        rt_score     STRING,
        popularity   FLOAT
    )
""")

push_df = movies_df[[
    "tmdb_id", "imdb_id", "title", "year",
    "genre", "region", "imdb_rating", "rt_score", "popularity"
]].reset_index(drop=True)

push_df.columns = [c.upper() for c in push_df.columns]
push_df["TMDB_ID"] = push_df["TMDB_ID"].astype(str)
push_df["YEAR"] = push_df["YEAR"].astype(str)

write_pandas(conn, push_df, "MOVIES_MASTER",
             database="MOVIE_ANALYTICS", schema="RAW")
print(f"Pushed {len(push_df)} movies to Snowflake!")
conn.close()

In [ ]:
# Re-fetch Malayalam movies only
def get_movies_by_language(language, region, years=[2023, 2024, 2025]):
    movies = []
    for year in years:
        for page in range(1, 6):
            url = "https://api.themoviedb.org/3/discover/movie"
            params = {
                "api_key": TMDB_KEY,
                "with_original_language": language,
                "primary_release_year": year,
                "sort_by": "popularity.desc",
                "page": page
            }
            r = requests.get(url, params=params).json()
            if not r.get("results"):
                break
            for movie in r["results"]:
                movies.append({
                    "title":      movie["title"],
                    "year":       year,
                    "region":     region,
                    "tmdb_id":    movie["id"],
                    "popularity": movie.get("popularity", 0)
                })
            time.sleep(0.1)
    return movies

print("Fetching Mollywood movies...")
mollywood = get_movies_by_language("ml", "Mollywood")
mollywood_df = pd.DataFrame(mollywood)
print(f"Found {len(mollywood_df)} Malayalam movies")
mollywood_df.head()

In [ ]:
# Step 1 - Get IMDb IDs
print("Getting IMDb IDs...")
imdb_ids = []
for i, row in mollywood_df.iterrows():
    url = f"https://api.themoviedb.org/3/movie/{row["tmdb_id"]}/external_ids"
    r = requests.get(url, params={"api_key": TMDB_KEY}).json()
    imdb_ids.append(r.get("imdb_id"))
    time.sleep(0.1)

mollywood_df["imdb_id"] = imdb_ids
mollywood_df = mollywood_df[mollywood_df["imdb_id"].notna()]
print(f"{len(mollywood_df)} movies with IMDb IDs")

# Step 2 - Get OMDB ratings
print("Getting IMDb + RT ratings...")
imdb_ratings, rt_scores, genres = [], [], []

for i, row in mollywood_df.iterrows():
    url = f"http://www.omdbapi.com/?i={row["imdb_id"]}&apikey={OMDB_KEY}"
    r = requests.get(url).json()
    
    imdb = r.get("imdbRating")
    imdb = float(imdb) if imdb and imdb != "N/A" else None
    rt = next((x["Value"] for x in r.get("Ratings", [])
               if x["Source"] == "Rotten Tomatoes"), None)
    genre = r.get("Genre")
    
    imdb_ratings.append(imdb)
    rt_scores.append(rt)
    genres.append(genre)
    time.sleep(0.2)

mollywood_df["imdb_rating"] = imdb_ratings
mollywood_df["rt_score"]    = rt_scores
mollywood_df["genre"]       = genres

mollywood_df = mollywood_df[mollywood_df["imdb_rating"].notna()]
print(f"{len(mollywood_df)} Mollywood movies with valid ratings")
mollywood_df.head(10)

In [ ]:
print(f"Total Malayalam movies: {len(mollywood_df)}")
print(f"With IMDb ID: {mollywood_df['imdb_id'].notna().sum()}")
print(f"With IMDb rating: {mollywood_df['imdb_rating'].notna().sum()}")

# Show sample
mollywood_df[["title", "imdb_id", "imdb_rating"]].head(10)

In [ ]:
# Search for known big Malayalam movies directly
big_movies = ["Manjummel Boys", "Premalu", "Aavesham", "Amaran", "Bramayugam"]

for movie in big_movies:
    r = requests.get(f"http://www.omdbapi.com/?t={movie}&apikey={OMDB_KEY}").json()
    print(f"{movie} → IMDb: {r.get("imdbRating")} | Response: {r.get("Response")}")

In [ ]:
# Manjummel Boys IMDb ID is tt27564649
r = requests.get(f"http://www.omdbapi.com/?i=tt27564649&apikey={OMDB_KEY}").json()
print(r)